# Results Notebook

Code here replicates graphs presented in the report. Follow instructions under load data
to determine which experiment to replicate and note any futher instructions through
notebook.

In [ ]:

import json
from pathlib import Path

import pandas as pd
from plotnine import (
    aes,
    element_text,
    facet_grid,
    facet_wrap,
    geom_label,
    geom_point,
    geom_smooth,
    geom_tile,
    ggplot,
    labs,
    theme,
    theme_bw,
)

from dataset_similarity.constants import (
    DATA_CONFIG_DIR,
    EVAL_RESULT_DIR,
    EXPERIMENT_CONFIG_DIR,
    METRICS_RESULT_DIR,
    PROJECT_DIR,
)
from dataset_similarity.utils import load_yaml_from_path

In [ ]:

UNLABELLED_METRIC_LABEL_MAP = {
    "mmd": "MMD",
    "ot_exact": "OT Exact",
    "ot_sinkhorn": "OT Sinkhorn",
}

LABELLED_METRIC_LABEL_MAP = {
    "otce_ot_sinkhorn_coupling": "OTCE Sinkhorn Coupling",
    "otdd_approx": "OTDD Approx",
}

DV_LABEL_MAP = {
    "accuracy_difference": "Accuracy Difference",
    "average_precision_difference": "Average Precision Difference",
    "precision_difference": "Precision Difference",
    "recall_difference": "Recall Difference",
    "f1_difference": "F1 Difference",
    "roc_auc_difference": "ROC AUC Difference",
}

## Load data

First, fill the list here with the experiment names you would like to load and produce
results for.

Comment out or add in a new set of experiment names to determine which experiment(s) to
produce plots for.

In [ ]:
experiment_names = ["experiment_1_main"]
# experiment_names = ["experiment_2_balance"]

# The following determines the output name
if len(experiment_names) == 1:
    output_name = experiment_names[0]
else:
    output_name = "_".join(experiment_names)

Next, run the following cells

In [ ]:
def load_json(result_path):
    """Wrapper around json.load"""
    with open(result_path) as f:
        return json.load(f)


def get_task_id(task_name: str) -> int:
    """Get the task id from the task name"""
    return int(task_name.split("_")[-1])


def load_experiment_results(experiment_name) -> pd.DataFrame:
    eval_results = pd.DataFrame(
        load_json(path)
        for path in (EVAL_RESULT_DIR / experiment_name).glob("finetune_*_results.json")
    )
    metrics_results = pd.DataFrame(
        load_yaml_from_path(path)
        for path in list((METRICS_RESULT_DIR / experiment_name).glob("metrics_*.yaml"))
    )
    eval_results["id"] = eval_results.name.apply(get_task_id)
    metrics_results["id"] = metrics_results.dataset1.apply(get_task_id)
    results = pd.merge(eval_results, metrics_results, on="id", how="inner")
    results = results.sort_values(by="id").reset_index(drop=True)
    return results


def load_metrics_used(experiment_names: list[str]) -> list[str]:
    metrics = []
    for experiment_name in experiment_names:
        cfg = load_yaml_from_path(EXPERIMENT_CONFIG_DIR / f"{experiment_name}.yaml")
        metrics += cfg["metrics"]
    return list(set(metrics))


def load_all_results(experiment_names: list[str]) -> tuple[pd.DataFrame, list[str]]:
    return pd.concat(
        [load_experiment_results(name) for name in experiment_names],
        ignore_index=True,
    ), load_metrics_used(experiment_names)


def melt_to_plot_df(results_df: pd.DataFrame, metrics: list[str]) -> pd.DataFrame:
    eval_metrics = results_df.columns[results_df.columns.str.endswith("_difference")]
    return results_df.melt(
        id_vars=["id", "name", *eval_metrics],
        value_vars=metrics,
        var_name="metric",
        value_name="value"
    ).melt(
        id_vars=["id", "name", "metric", "value"],
        value_vars=eval_metrics,
        var_name="dv",
        value_name="dv_value"
    ).assign(
        metric=lambda x: x.metric.map({**UNLABELLED_METRIC_LABEL_MAP, **LABELLED_METRIC_LABEL_MAP}),
    ).assign(
        dv=lambda x: x.dv.map(DV_LABEL_MAP),
    )


def load_task_attributes(task_path: Path) -> dict:
    cfg = load_yaml_from_path(task_path)
    kwargs = cfg["kwargs"]
    kwargs["name"] = (
        task_path.parent.stem + "/" + task_path.stem.replace("testARC", "finetune")
    )
    return {
        k: v[0] if isinstance(v, list) and len(v) == 1 else v for k, v in kwargs.items()
    }


def load_experiment_task_attributes(experiment_name: str) -> list[dict]:
    task_paths = (DATA_CONFIG_DIR / experiment_name).glob("testARC_*.yaml")
    return pd.DataFrame(load_task_attributes(path) for path in task_paths)

In [ ]:
results, metrics = load_all_results(experiment_names)
task_data = pd.concat(
    load_experiment_task_attributes(name) for name in experiment_names
)

In [ ]:

pdf = melt_to_plot_df(results, metrics)
pdf = pd.merge(pdf, task_data, on="name", how="left")

## Plot Metric Performances for AP Diff Only

In [ ]:
def plot_metrics_vs_ap_difference(pdf: pd.DataFrame, use_unlabelled=False, use_labelled=False):
    metric_labels = []
    if use_unlabelled:
        metric_labels += list(UNLABELLED_METRIC_LABEL_MAP.values())
    if use_labelled:
        metric_labels += list(LABELLED_METRIC_LABEL_MAP.values())
    if len(metric_labels) == 0:
        raise ValueError("At least one of use_unlabelled or use_labelled must be True")
    pdf = pdf[pdf.metric.isin(metric_labels)]
    return (
        ggplot(
            pdf[pdf.dv == "Average Precision Difference"],
            aes(x="value", y="dv_value"),
        )
        + geom_point()
        + geom_smooth(method="lm", color="red")
        + facet_wrap("~metric", scales="free")
        + labs(
            x="Metric Value",
            y="Average Precision Difference",
        )
        + theme_bw()
        + theme(
            axis_text_x=element_text(rotation=45, hjust=1),
            figure_size=(8, 4) if use_unlabelled and not use_labelled else (6, 4),
            aspect_ratio=1.0,
        )
    )

In [ ]:
p = plot_metrics_vs_ap_difference(pdf, use_unlabelled=True)
p.save(PROJECT_DIR / f"plots/{output_name}_metrics_vs_ap_difference_unlabelled.png", dpi=300)
p

In [ ]:
p = plot_metrics_vs_ap_difference(pdf, use_labelled=True)
p.save(PROJECT_DIR / f"plots/{output_name}_metrics_vs_ap_difference_labelled.png", dpi=300)
p

## Optionally Add Colour

In [ ]:
def plot_metrics_vs_ap_difference_colour(
    pdf: pd.DataFrame,
    group: str,
    use_unlabelled=False,
    use_labelled=False,
):
    metric_labels = []
    if use_unlabelled:
        metric_labels += list(UNLABELLED_METRIC_LABEL_MAP.values())
    if use_labelled:
        metric_labels += list(LABELLED_METRIC_LABEL_MAP.values())
    if len(metric_labels) == 0:
        raise ValueError("At least one of use_unlabelled or use_labelled must be True")
    pdf = pdf[pdf.metric.isin(metric_labels)]
    return (
        ggplot(
            pdf[pdf.dv == "Average Precision Difference"],
            aes(x="value", y="dv_value"),
        )
        + geom_point(aes(color=group, shape=group))
        + geom_smooth(method="lm", color="black", se=True)
        + facet_wrap("~metric", scales="free", ncol=2)
        + labs(
            x="Metric Value",
            y="Average Precision Difference",
        )
        + theme_bw()
        + theme(
            axis_text_x=element_text(rotation=45, hjust=1),
            legend_position=(1.,0.) if use_unlabelled and not use_labelled else "right",
            figure_size=(6, 6) if use_unlabelled and not use_labelled else (6, 4),
            aspect_ratio=1.0,
        )
    )

In [ ]:
p = plot_metrics_vs_ap_difference_colour(pdf, group="positive_class", use_unlabelled=True)
p.save(PROJECT_DIR / f"plots/{output_name}_metrics_vs_ap_difference_unlabelled_positive_class.png", dpi=300)
p

In [ ]:
p = plot_metrics_vs_ap_difference_colour(pdf, group="positive_class", use_labelled=True)
p.save(PROJECT_DIR / f"plots/{output_name}_metrics_vs_ap_difference_labelled_positive_class.png", dpi=300)
p

## Result Grid

In [ ]:
def plot_metrics_vs_dv(pdf: pd.DataFrame, use_unlabelled=False, use_labelled=False):
    metric_labels = []
    if use_unlabelled:
        metric_labels += list(UNLABELLED_METRIC_LABEL_MAP.values())
    if use_labelled:
        metric_labels += list(LABELLED_METRIC_LABEL_MAP.values())
    if len(metric_labels) == 0:
        raise ValueError("At least one of use_unlabelled or use_labelled must be True")
    pdf = pdf[pdf.metric.isin(metric_labels)]
    return (
        ggplot(
            pdf,
            aes(x="value", y="dv_value"),
        )
        + geom_point()
        + geom_smooth(method="lm", color="red")
        + facet_grid("dv~metric", scales="free")
        + labs(
            x="Metric Value",
            y="Average Precision Difference",
        )
        + theme_bw()
        + theme(
            axis_text_x=element_text(rotation=45, hjust=1),
            figure_size=(6, 10),
            # aspect_ratio=1.0,
        )
    )

In [ ]:
p = plot_metrics_vs_dv(pdf, use_unlabelled=True)
p.save(PROJECT_DIR / f"plots/{output_name}_metrics_vs_outcome_grid_unlabelled_positive_class.png", dpi=300)
p

In [ ]:
p = plot_metrics_vs_dv(pdf, use_labelled=True)
p.save(PROJECT_DIR / f"plots/{output_name}_metrics_vs_outcome_grid_labelled_positive_class.png", dpi=300)
p

## Correlations

In [ ]:
corr_vars = {
    "average_precision_difference": "Average Precision Difference",
    **UNLABELLED_METRIC_LABEL_MAP,
    **LABELLED_METRIC_LABEL_MAP,
}

def build_corr_plot(df, corr="pearson"):
    vars = {k: v for k, v in corr_vars.items() if k in df.columns}
    corr_pdf = (
        df[vars.keys()]
        .rename(columns=vars)
        .corr(corr)
        .melt(ignore_index=False)
        .reset_index()
        .set_axis(["metric_1", "metric_2", "correlation"], axis=1)
        .assign(lab_text=lambda x: x.correlation.map(lambda v: f"{v:.2f}"))
    )
    corr_pdf.metric_1 = pd.Categorical(
        corr_pdf.metric_1,
        categories=vars.values(),
        ordered=True,
    )
    corr_pdf.metric_2 = pd.Categorical(
        corr_pdf.metric_2,
        categories=reversed(vars.values()),
        ordered=True,
    )
    title = "Pearson's Correlation Matrix" if corr == "pearson" else "Spearman's Correlation Matrix"
    return (
        ggplot(
            corr_pdf,
            aes(
                x="metric_1",
                y="metric_2",
                fill="correlation",
                label="lab_text",
            )
        )
        + geom_tile()
        + geom_label(fill="white", size=8)
        + labs(title=title, x="", y="")
        + theme_bw()
        + theme(
            axis_text_x = element_text(rotation=45, hjust=1),
            aspect_ratio=1,
        )
    )

In [ ]:
p = build_corr_plot(results, "pearson")
p.save(PROJECT_DIR / f"plots/{output_name}_pearson_corr.png", dpi=300)
p

In [ ]:
build_corr_plot(results, "spearman")
p.save(PROJECT_DIR / f"plots/{output_name}_spearmans_corr.png", dpi=300)
p